In [1]:
import numpy as np
import bayesflow as bf
import keras
from pathlib import Path
import pickle
from benchmark.examples.gaussian.direct.simulator import DirectSimulator   
from benchmark.examples.gaussian.direct.approximators import DirectBayesFLowNPE

INFO:bayesflow:Using backend 'jax'


In [2]:
RNG=np.random.default_rng(2025)
num_dims=5
num_obs=100
likelihood_std=1

In [ ]:
simulator_direct=DirectSimulator( num_dims=num_dims, num_obs=num_obs,
                                likelihood_std=likelihood_std,rng=RNG)
simulator=simulator_direct.get_direct_simulator()
bayesflownpe=DirectBayesFLowNPE(simulator=simulator,num_batches_per_epoch=50,batch_size=100,epochs=64)
history=bayesflownpe.train()

In [3]:
# Load the trained approximator
filepath = Path("/Users/mandyking/benchmark/benchmark/examples/gaussian/networks") / "direct.keras"
approximator=keras.saving.load_model(filepath)
simulator_direct=DirectSimulator( num_dims=num_dims, num_obs=num_obs,
                                likelihood_std=likelihood_std,rng=RNG)
simulator=simulator_direct.get_direct_simulator()
bayesflownpe=DirectBayesFLowNPE(simulator=simulator,num_batches_per_epoch=50,batch_size=100,epochs=64)


In [ ]:
df=simulator.sample(5000)
pred_models = approximator.predict(conditions=df, probs=True)

In [ ]:
f = bf.diagnostics.plots.mc_confusion_matrix(
    pred_models=pred_models,
    true_models=df["model_indices"],
    model_names=[r"$\mathcal{M}_1$", r"$\mathcal{M}_2$",r"$\mathcal{M}_3$"],
    normalize="true",
)

In [ ]:
# Save the trained approximator
filepath = Path("/Users/mandyking/benchmark/benchmark/examples/gaussian/networks") / "direct.keras"
filepath.parent.mkdir(exist_ok=True)
bayesflownpe.workflow.save(filepath=filepath)

In [4]:
# read datasets
def load_pickle(path: str):
    with open(path, "rb") as f:
        return pickle.load(f)

datasets_normal_0  = load_pickle("/Users/mandyking/benchmark/benchmark/examples/gaussian/results/datasets/01datasets_normal_0.pkl")
datasets_normal_10 = load_pickle("/Users/mandyking/benchmark/benchmark/examples/gaussian/results/datasets/01datasets_normal_10.pkl")
datasets_student   = load_pickle("/Users/mandyking/benchmark/benchmark/examples/gaussian/results/datasets/01datasets_student_df10.pkl")

In [5]:
datasets_normal_0=bayesflownpe.get_probs_M1(datasets_normal_0,approximator)
datasets_normal_10=bayesflownpe.get_probs_M2(datasets_normal_10,approximator)
datasets_student=bayesflownpe.get_probs_M3(datasets_student,approximator)

In [8]:
print(datasets_student[0].keys())
print(datasets_normal_0[0]["pred_model"])


dict_keys(['mu', 'obs_data', 'id', 'df', 'gold_log_marginal', 'gold_post_samples', 'npe_post_samples', 'npe_log_marginal', 'pred_model', 'logBF_31_direct', 'logBF_32_direct', 'logBF_12_direct'])
[[5.0375670e-01 5.6259373e-06 4.9623767e-01]]


In [9]:
# save datasetst
def save_pickle(obj, path: str):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "wb") as f:
        pickle.dump(obj, f)

save_pickle(datasets_normal_0,  "/Users/mandyking/benchmark/benchmark/examples/gaussian/results/datasets/04datasets_normal_0.pkl")
save_pickle(datasets_normal_10, "/Users/mandyking/benchmark/benchmark/examples/gaussian/results/datasets/04datasets_normal_10.pkl")
save_pickle(datasets_student,   "/Users/mandyking/benchmark/benchmark/examples/gaussian/results/datasets/04datasets_student_df10.pkl")